# Member 1 — Linguistic Error Analysis
**Project:** Revisiting MARBERT for Arabic Sentiment through Reproducibility and Linguistic Robustness (AI504, Assignment 2)

**What this notebook does:** reads the saved test predictions of MARBERT and AraBERT (from Members 2 and 3) and produces:
1. scores per run (accuracy, macro-F1, F1-PN, per-class precision / recall / F1 / support);
2. confusion matrices (which classes are mixed up);
3. results per **dialect** and for **sarcastic vs non-sarcastic** tweets (ArSarcasm), always with counts;
4. the most common error types;
5. a list of **candidate failure examples** to choose 2–3+ from for the report.

**Inputs (upload before running):**
- `Team_Splits_Final.zip` — the final team split files (upload it; the notebook unzips it).
- `predictions/` — the results folder of Members 2 and 3 (e.g. `outputs/historical/`, zipped or not), with one
  `epoch_XX_test...csv` file per epoch (columns idx, gold, pred, ...). The notebook picks the best epoch itself.
  OR one or more CSV files with the columns:
  `dataset` (arsarcasm / astd / ajgt), `model` (MARBERT / AraBERT), `protocol` (historical / independent),
  `seed`, `id` (the id from the split files), `gold`, `pred` (negative / neutral / positive).

If `predictions/` is empty, the notebook runs in **DEMO mode** with random fake predictions, only to show that it works.

In [ ]:
# ---------- 1. Settings and loading ----------
import os, glob, zipfile
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, accuracy_score, precision_recall_fscore_support, confusion_matrix
import matplotlib.pyplot as plt

SPLITS = "member1_outputs/splits"
PRED = "predictions"
OUT = "member1_error_analysis"
LABELS = ["negative", "neutral", "positive"]
os.makedirs(PRED, exist_ok=True); os.makedirs(OUT, exist_ok=True)

# Unzip the team split files if they are not unzipped yet (Team_Splits_Final.zip or member1_outputs.zip)
for zname in ["Team_Splits_Final.zip", "member1_outputs.zip"]:
    if not os.path.isdir(SPLITS) and os.path.exists(zname):
        zipfile.ZipFile(zname).extractall(".")
        print("Unzipped", zname)
# Unzip any zipped prediction files placed in predictions/
for zf in glob.glob(os.path.join(PRED, "*.zip")):
    zipfile.ZipFile(zf).extractall(PRED)
assert os.path.isdir(SPLITS), "Upload Team_Splits_Final.zip first (Files panel on the left)."
tests = {ds: pd.read_csv(os.path.join(SPLITS, f"{ds}_test.csv"), dtype={"id": str}) for ds in ["arsarcasm", "astd", "ajgt"]}
for ds, t in tests.items():
    print(f"{ds}: {len(t)} test tweets")

files = sorted(glob.glob(os.path.join(PRED, "**", "*.csv"), recursive=True))
DEMO = len(files) == 0
if DEMO:
    print("\n*** DEMO MODE: no prediction files found, using random fake predictions. Do NOT report these numbers. ***\n")
    rng = np.random.default_rng(0); rows = []
    for ds, t in tests.items():
        labs = sorted(t["label"].unique())
        for model, acc in [("MARBERT", 0.75), ("AraBERT", 0.68)]:
            for protocol in ["historical", "independent"]:
                for seed in [1, 2, 3]:
                    pred = np.where(rng.random(len(t)) < acc, t["label"], rng.choice(labs, len(t)))
                    rows.append(pd.DataFrame({"dataset": ds, "model": model, "protocol": protocol, "seed": seed,
                                              "id": t["id"], "gold": t["label"], "pred": pred}))
    preds = pd.concat(rows, ignore_index=True)
else:
    import re, json
    std = [f for f in files if {"dataset", "model", "seed", "id", "gold", "pred"} <= set(c.strip().lower() for c in pd.read_csv(f, nrows=0).columns)]
    epoch_files = [f for f in files if re.search(r"epoch_?(\d+)", os.path.basename(f), re.I) and f not in std]
    parts = [pd.read_csv(f, dtype={"id": str}) for f in std]

    # ---- Member 2 / Member 3 notebook format: one CSV per epoch with columns idx, gold, pred, p_negative, ...
    #      (folder per run; dataset, model and seed are read from run_info.json or from the folder names)
    def run_meta(path):
        meta, d = {}, os.path.dirname(path)
        for _ in range(4):                                   # look for run_info.json in this folder and up to 3 parents
            p = os.path.join(d, "run_info.json")
            if os.path.exists(p):
                try: meta = {k.lower(): v for k, v in json.load(open(p)).items()}
                except Exception: pass
                break
            d = os.path.dirname(d)
        low = path.lower()
        ds = str(meta.get("dataset", meta.get("target", ""))).lower()
        ds = next((x for x in ["arsarcasm", "astd", "ajgt"] if x in ds or x in low.replace("_", "").replace("-", "")), None)
        mod = str(meta.get("model", meta.get("model_name", ""))).lower() + " " + low
        mod = "MARBERT" if "marbert" in mod else ("AraBERT" if "arabert" in mod else None)
        seed = meta.get("seed")
        if seed is None:
            m = re.search(r"seed[_-]?(\d+)", low) or re.search(r"(?:^|[/_-])(4[2-9])(?:[/_-]|$)", low)
            seed = int(m.group(1)) if m else None
        prot = "independent" if ("independent" in low or "dev_selected" in low) else "historical"
        return ds, mod, seed, prot

    def label_map(ds, gold_codes):
        """Find which number means which label by comparing the gold codes with the true test labels."""
        t = tests[ds]["label"].values
        names = sorted(set(t))                               # alphabetical: negative, neutral, positive
        for perm in [names, ["negative", "neutral", "positive"], ["negative", "positive"], ["positive", "negative"],
                     ["negative", "positive", "neutral"], ["neutral", "positive", "negative"]]:
            if len(perm) != len(names): continue
            mapped = pd.Series(gold_codes).map(dict(enumerate(perm))).values
            if len(mapped) == len(t) and (mapped == t).mean() > 0.999:
                return dict(enumerate(perm))
        raise ValueError(f"{ds}: cannot match the numeric gold labels to the test file. Are the rows in the same order?")

    def score(ds, g, p):
        if ds == "ajgt": return accuracy_score(g, p)
        return f1_score(g, p, labels=["positive", "negative"], average="macro", zero_division=0)   # F1-PN, as in the paper

    runs = {}
    for f in epoch_files:
        if re.search(r"[/\\](smoke|debug|test_run)[/\\]", f.lower()):   # skip quick smoke-test runs
            continue
        ds, mod, seed, prot = run_meta(f)
        split = "dev" if re.search(r"dev", os.path.basename(f), re.I) else "test"
        ep = int(re.search(r"epoch_?(\d+)", os.path.basename(f), re.I).group(1))
        if None in (ds, mod, seed):
            print("Skipped (cannot tell dataset/model/seed):", f); continue
        runs.setdefault((ds, mod, seed, prot), {}).setdefault(split, {})[ep] = f

    chosen = []
    for (ds, mod, seed, prot), sp in sorted(runs.items(), key=lambda x: str(x[0])):
        if "test" not in sp: continue
        t = tests[ds]
        res = {}
        info_labels = None
        rid = os.path.join(os.path.dirname(os.path.dirname(sp["test"][min(sp["test"])])), "run_info.json")
        if os.path.exists(rid):
            info_labels = json.load(open(rid)).get("labels")
        for ep, f in sorted(sp["test"].items()):
            e = pd.read_csv(f)
            mp = dict(enumerate(info_labels)) if info_labels else label_map(ds, e["gold"].values)
            assert (e["gold"].map(mp).values == t["label"].values[e["idx"].values if "idx" in e.columns else np.arange(len(e))]).mean() > 0.999, \
                f"{f}: gold labels do not match the test file"
            g = e["gold"].map(mp).values; p = e["pred"].map(mp).values
            res[ep] = (score(ds, g, p), g, p, e["idx"].values if "idx" in e.columns else np.arange(len(e)))
        if prot == "independent" and "dev" in sp:            # independent protocol: best epoch on DEV macro-F1
            dev_sc = {}
            for ep, f in sp["dev"].items():
                e = pd.read_csv(f); mp = {i: l for i, l in enumerate(sorted(set(t["label"])))}
                dev_sc[ep] = f1_score(e["gold"].map(mp), e["pred"].map(mp), average="macro", zero_division=0)
            best = max(dev_sc, key=dev_sc.get)
        else:                                                # historical protocol: best epoch on TEST (paper's method)
            best = max(res, key=lambda k: res[k][0])
        sc, g, p, idx = res[best]
        chosen.append({"dataset": ds, "model": mod, "protocol": prot, "seed": seed, "best_epoch": best,
                       "score_paper_metric": round(sc * 100, 2), "epochs_found": len(res)})
        parts.append(pd.DataFrame({"dataset": ds, "model": mod, "protocol": prot, "seed": seed,
                                   "id": t["id"].values[idx], "gold": g, "pred": p}))
    if chosen:
        chosen_df = pd.DataFrame(chosen); chosen_df.to_csv(os.path.join(OUT, "selected_epochs.csv"), index=False)
        print("Runs read from per-epoch files (best epoch chosen automatically):")
        try: display(chosen_df)
        except NameError: print(chosen_df.to_string())
    assert parts, "No usable prediction files found in predictions/"
    preds = pd.concat(parts, ignore_index=True)
    print("Loaded", len(files), "files ->", len(preds), "prediction rows")

# ---- checks: every prediction must match a test tweet and its gold label ----
preds.columns = [c.strip().lower() for c in preds.columns]
need = {"dataset", "model", "protocol", "seed", "id", "gold", "pred"}
assert need <= set(preds.columns), f"Missing columns: {need - set(preds.columns)}"
for c in ["dataset", "protocol", "gold", "pred"]:
    preds[c] = preds[c].astype(str).str.strip().str.lower()
preds["model"] = preds["model"].astype(str).str.strip().replace({"marbert": "MARBERT", "arabert": "AraBERT"})
unknown = sorted(set(preds["dataset"]) - set(tests))
if unknown:   # e.g. ajgt_ubc (extra split from Member 2): not part of the shared test files, so it is left out here
    print("Left out (no matching test file in Team_Splits_Final):", unknown)
    preds = preds[preds["dataset"].isin(tests)].copy()
problems = []
for ds, g in preds.groupby("dataset"):
    t = tests[ds].set_index("id")
    missing = ~g["id"].isin(t.index)
    wrong_gold = g.loc[~missing].apply(lambda r: t.at[r["id"], "label"] != r["gold"], axis=1)
    problems.append({"dataset": ds, "rows": len(g), "unknown_ids": int(missing.sum()), "gold_mismatch": int(wrong_gold.sum()),
                     "invalid_pred_labels": int((~g["pred"].isin(LABELS)).sum())})
pd.DataFrame(problems)

In [ ]:
# ---------- 2. Scores for every run ----------
def scores(g, ds):
    y, p = g["gold"], g["pred"]
    labs = ["negative", "positive"] if ds == "ajgt" else LABELS
    r = {"accuracy": accuracy_score(y, p), "macro_F1": f1_score(y, p, labels=labs, average="macro", zero_division=0)}
    if ds != "ajgt":   # F1-PN: mean of positive- and negative-class F1 on the full 3-class test set
        r["F1_PN"] = f1_score(y, p, labels=["positive", "negative"], average="macro", zero_division=0)
    pr, rc, f1, sup = precision_recall_fscore_support(y, p, labels=labs, zero_division=0)
    for l, a, b, c, d in zip(labs, pr, rc, f1, sup):
        r[f"{l}_P"], r[f"{l}_R"], r[f"{l}_F1"], r[f"{l}_support"] = a, b, c, int(d)
    return r

run_rows = []
for (ds, model, protocol, seed), g in preds.groupby(["dataset", "model", "protocol", "seed"]):
    run_rows.append({"dataset": ds, "model": model, "protocol": protocol, "seed": seed, "n": len(g), **scores(g, ds)})
runs = pd.DataFrame(run_rows)
runs.round(4).to_csv(os.path.join(OUT, "scores_per_run.csv"), index=False)

main_cols = ["accuracy", "macro_F1", "F1_PN"]
summary = (runs.groupby(["dataset", "protocol", "model"])[main_cols]
           .agg(["mean", "std"]).mul(100).round(2))
summary.columns = [f"{m}_{s}" for m, s in summary.columns]
summary.to_csv(os.path.join(OUT, "scores_summary_mean_sd.csv"))
summary

In [ ]:
# ---------- 3. Confusion matrices (independent protocol; average over seeds, % of each true class) ----------
PROTOCOL = "independent" if "independent" in set(preds["protocol"]) else sorted(preds["protocol"].unique())[0]
models = sorted(preds["model"].unique(), key=lambda m: (m != "MARBERT", m))
for ds in ["arsarcasm", "astd", "ajgt"]:
    g = preds[(preds.dataset == ds) & (preds.protocol == PROTOCOL)]
    if g.empty:
        continue
    labs = ["negative", "positive"] if ds == "ajgt" else LABELS
    fig, axes = plt.subplots(1, len(models), figsize=(4.2 * len(models), 3.6))
    axes = np.atleast_1d(axes)
    for ax, model in zip(axes, models):
        gm = g[g.model == model]
        cm = confusion_matrix(gm["gold"], gm["pred"], labels=labs)
        n_seeds = gm["seed"].nunique()
        pct = cm / cm.sum(axis=1, keepdims=True).clip(min=1) * 100
        ax.imshow(pct, cmap="Blues", vmin=0, vmax=100)
        for i in range(len(labs)):
            for j in range(len(labs)):
                ax.text(j, i, f"{pct[i, j]:.0f}%\n({cm[i, j] / n_seeds:.0f})", ha="center", va="center",
                        fontsize=9, color="white" if pct[i, j] > 60 else "#1f1f1f")
        ax.set_xticks(range(len(labs))); ax.set_xticklabels(labs); ax.set_yticks(range(len(labs))); ax.set_yticklabels(labs)
        ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(f"{model} ({n_seeds} seeds)")
    fig.suptitle(f"{ds}: confusion matrix, {PROTOCOL} protocol (row %, mean count)" + ("  [DEMO]" if DEMO else ""), fontsize=10)
    plt.tight_layout(); plt.savefig(os.path.join(OUT, f"confusion_{ds}.png"), dpi=200); plt.show()

In [ ]:
# ---------- 4. ArSarcasm: results per dialect and for sarcastic vs non-sarcastic tweets ----------
meta = tests["arsarcasm"][["id", "text", "dialect", "sarcasm"]]
ag = preds[(preds.dataset == "arsarcasm") & (preds.protocol == PROTOCOL)].merge(meta, on="id")
ag["sarcasm"] = ag["sarcasm"].astype(str).str.lower().map({"true": "sarcastic", "false": "non-sarcastic"})

def subgroup_table(col):
    rows = []
    for (grp, model, seed), g in ag.groupby([col, "model", "seed"]):
        rows.append({col: grp, "model": model, "seed": seed, "n": len(g),
                     "macro_F1": f1_score(g.gold, g.pred, labels=LABELS, average="macro", zero_division=0) * 100,
                     "F1_PN": f1_score(g.gold, g.pred, labels=["positive", "negative"], average="macro", zero_division=0) * 100,
                     "accuracy": accuracy_score(g.gold, g.pred) * 100})
    t = pd.DataFrame(rows).groupby([col, "model"]).agg(n=("n", "first"), macro_F1_mean=("macro_F1", "mean"),
            macro_F1_sd=("macro_F1", "std"), F1_PN_mean=("F1_PN", "mean"), accuracy_mean=("accuracy", "mean")).round(2)
    t["note"] = np.where(t["n"] < 100, "small group: exploratory", "")
    return t.reset_index()

by_dialect = subgroup_table("dialect"); by_sarcasm = subgroup_table("sarcasm")
by_dialect.to_csv(os.path.join(OUT, "arsarcasm_by_dialect.csv"), index=False)
by_sarcasm.to_csv(os.path.join(OUT, "arsarcasm_by_sarcasm.csv"), index=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), gridspec_kw={"width_ratios": [5, 2]})
for ax, t, col in [(axes[0], by_dialect, "dialect"), (axes[1], by_sarcasm, "sarcasm")]:
    groups = list(dict.fromkeys(t[col])); x = np.arange(len(groups)); w = 0.8 / len(models)
    for i, model in enumerate(models):
        tm = t[t.model == model].set_index(col).reindex(groups)
        bars = ax.bar(x + (i - (len(models) - 1) / 2) * w, tm["macro_F1_mean"], w, yerr=tm["macro_F1_sd"].fillna(0),
                      capsize=2, label=model, color=["#2F6FD0", "#E8673C", "#2E9E57"][i % 3], edgecolor="white")
        ax.bar_label(bars, fmt="%.0f", fontsize=7, padding=2)
    ns = t.drop_duplicates(col).set_index(col)["n"].reindex(groups)
    ax.set_xticks(x); ax.set_xticklabels([f"{g}\n(n={n})" for g, n in zip(groups, ns)], fontsize=8)
    ax.set_ylim(0, 105); ax.set_ylabel("Macro-F1 (mean ± SD over seeds)" if col == "dialect" else "")
    ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="y", alpha=0.25)
axes[0].set_title("ArSarcasm by dialect"); axes[1].set_title("ArSarcasm by sarcasm")
axes[0].legend(frameon=False, fontsize=8, ncol=len(models), loc="upper left", bbox_to_anchor=(0, 1.0))
axes[0].set_ylim(0, 115); axes[1].set_ylim(0, 115)
if DEMO: fig.suptitle("[DEMO]", fontsize=9)
plt.tight_layout(); plt.savefig(os.path.join(OUT, "arsarcasm_dialect_sarcasm.png"), dpi=200); plt.show()
by_dialect

In [ ]:
# ---------- 5. Most common error types (true -> predicted), independent protocol ----------
err = preds[(preds.protocol == PROTOCOL) & (preds.gold != preds.pred)]
err_types = err.groupby(["dataset", "model", "gold", "pred"]).size()
n_seeds = preds[preds.protocol == PROTOCOL].groupby(["dataset", "model"])["seed"].nunique()
err_types = err_types.rename("errors_all_seeds").reset_index()
err_types["mean_errors_per_run"] = err_types.apply(lambda r: r["errors_all_seeds"] / n_seeds[(r["dataset"], r["model"])], axis=1).round(1)
err_types["error_type"] = err_types["gold"] + " -> " + err_types["pred"]
err_types = err_types.sort_values(["dataset", "model", "errors_all_seeds"], ascending=[True, True, False])
err_types.to_csv(os.path.join(OUT, "error_types.csv"), index=False)
err_types.groupby(["dataset", "model"]).head(3)[["dataset", "model", "error_type", "mean_errors_per_run"]]

In [ ]:
# ---------- 6. Candidate failure examples (to choose 2–3+ for the report) ----------
# For each test tweet: how many runs of each model got it wrong. "Always wrong" = wrong in every seed.
p = preds[preds.protocol == PROTOCOL].copy()
p["wrong"] = (p.gold != p.pred).astype(int)
stat = p.groupby(["dataset", "id", "model"]).agg(wrong_runs=("wrong", "sum"), runs=("wrong", "size"),
                                                 most_common_pred=("pred", lambda s: s.mode().iloc[0])).reset_index()
wide = stat.pivot_table(index=["dataset", "id"], columns="model", values=["wrong_runs", "runs", "most_common_pred"], aggfunc="first")
wide.columns = [f"{a}_{b}" for a, b in wide.columns]; wide = wide.reset_index()
allmeta = pd.concat([t.assign(dataset=ds) for ds, t in tests.items()], ignore_index=True)
cols = [c for c in ["dataset", "id", "text", "label", "dialect", "sarcasm"] if c in allmeta.columns]
cand = wide.merge(allmeta[cols], on=["dataset", "id"], how="left")

def category(r):
    mw = r.get("wrong_runs_MARBERT", 0) == r.get("runs_MARBERT", -1)
    aw = r.get("wrong_runs_AraBERT", 0) == r.get("runs_AraBERT", -1)
    mr = r.get("wrong_runs_MARBERT", 1) == 0
    ar = r.get("wrong_runs_AraBERT", 1) == 0
    if mw and aw: return "1. both models always wrong"
    if mw and ar: return "2. only MARBERT wrong (AraBERT always right)"
    if aw and mr: return "3. only AraBERT wrong (MARBERT always right)"
    return ""
cand["category"] = cand.apply(category, axis=1)
cand = cand[cand.category != ""].sort_values(["category", "dataset"])
cand["words"] = cand["text"].astype(str).str.split().str.len()
cand = cand[(cand.words >= 3) & (cand.words <= 40)]            # readable length for a report example
cand.to_csv(os.path.join(OUT, "failure_example_candidates.csv"), index=False, encoding="utf-8-sig")
print(cand["category"].value_counts().to_string())
cand.groupby("category").head(5)[["category", "dataset", "id", "text", "label", "most_common_pred_MARBERT", "most_common_pred_AraBERT"] +
                                  [c for c in ["dialect", "sarcasm"] if c in cand.columns]]

In [ ]:
# ---------- 7. Observations only (facts, no explanations) and a template for the failure examples ----------
lines = ["# Error analysis: observations" + ("  [DEMO: do not report]" if DEMO else ""), "",
         f"Protocol analysed: {PROTOCOL}. Values are means over seeds.", ""]
for (ds, protocol, model), r in summary.reset_index().set_index(["dataset", "protocol", "model"]).iterrows():
    if protocol == PROTOCOL:
        f1pn = f", F1-PN {r['F1_PN_mean']:.2f}" if not pd.isna(r.get("F1_PN_mean")) else ""
        lines.append(f"- {ds} / {model}: macro-F1 {r['macro_F1_mean']:.2f} (SD {r['macro_F1_std']:.2f}){f1pn}, accuracy {r['accuracy_mean']:.2f}.")
lines.append("")
for (ds, model), g in err_types.groupby(["dataset", "model"]):
    top = g.iloc[0]
    lines.append(f"- {ds} / {model}: most frequent error type is {top['error_type']} ({top['mean_errors_per_run']} per run).")
lines.append("")
for _, r in by_dialect.iterrows():
    lines.append(f"- ArSarcasm {r['dialect']} (n={r['n']}) / {r['model']}: macro-F1 {r['macro_F1_mean']:.2f}{' — ' + r['note'] if r['note'] else ''}.")
for _, r in by_sarcasm.iterrows():
    lines.append(f"- ArSarcasm {r['sarcasm']} (n={r['n']}) / {r['model']}: macro-F1 {r['macro_F1_mean']:.2f}.")
lines += ["", "# Failure examples (fill in 3 or more; observations first, then explanations)", ""]
for k in range(1, 4):
    lines += [f"## Example {k}", "- Tweet (Arabic): ", "- English translation: ", "- Dataset / dialect / sarcastic: ",
              "- True label: ", "- MARBERT predicted: ", "- AraBERT predicted: ",
              "- Observation (what we see): ", "- Possible explanation (what may cause it): ", ""]
with open(os.path.join(OUT, "observations_and_examples_template.md"), "w", encoding="utf-8") as f:
    f.write("\n".join(lines))
print("\n".join(lines[:25]))

In [ ]:
# ---------- 8. Zip the results ----------
zp = "member1_error_analysis.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for fn in sorted(os.listdir(OUT)):
        z.write(os.path.join(OUT, fn))
print("Created", zp)
try:
    from google.colab import files
    files.download(zp)
except ImportError:
    pass